# 三模型全量重训与秩集成

本 Notebook 是 `03_模型训练/y1_pipeline_v2/` 的唯一执行入口。它从头覆盖：

1. 线性 Mini-batch SGD（3 epochs）；
2. 486 期窗口 TCN（固定 8 epochs）；
3. LightGBM LambdaRank（固定 1558 轮）；
4. 同一批模型的 Valid/Test 推理；
5. Valid RankIC、三模型截面秩、权重搜索、半窗确认与候选文件生成。

默认 `FORCE_RETRAIN=False`：模型和预测产物存在且验收通过时直接加载，否则训练/推理。
若希望无条件重新训练三模型，请在下一单元设置 `FORCE_RETRAIN=True`。

**重要：**完整执行会进行长时间训练。本文件交付时只做过静态检查、合成数据测试和微型前向测试，
没有执行下面的正式训练单元。


## 1. 配置与依赖

必须从 `03_模型训练/y1_pipeline_v2/` 启动，并使用 `jingge_ts` 内核。
所有新模型、预测和报告只写入本目录的 `outputs/`，不会覆盖现有
`../y1_rank_pipeline/y1_rank_outputs/y1_best.npy`。


In [1]:
from __future__ import annotations

import gc
import json
import math
import random
import sys
import time
from dataclasses import asdict, dataclass
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from scipy.stats import rankdata

BASE_DIR = Path.cwd().resolve()
if BASE_DIR.name != "y1_pipeline_v2":
    raise RuntimeError(
        "请从 03_模型训练/y1_pipeline_v2/ 启动本 Notebook；"
        f"当前目录为 {BASE_DIR}"
    )

ROOT_DIR = BASE_DIR.parents[1]
RANK_PIPELINE_DIR = BASE_DIR.parent / "y1_rank_pipeline"
RANK_OUTPUT_DIR = RANK_PIPELINE_DIR / "y1_rank_outputs"
OUTPUT_DIR = BASE_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if str(RANK_PIPELINE_DIR) not in sys.path:
    sys.path.insert(0, str(RANK_PIPELINE_DIR))

from y1_rank_pipeline_lib import (  # noqa: E402
    CompetitionData,
    PipelineConfig,
    T,
    S,
    TRAIN_START,
    VALID_START,
    TEST_START,
    build_feature_matrix,
    build_history_cache,
    deterministic_stock_sample,
    fit_category_state,
    load_booster,
    open_history_cache,
    remove_feature_matrix,
    save_booster,
    train_ranker_no_validation,
)

SEED = 42
FORCE_RETRAIN = False

TRAIN_RANGE = (486, 2918)
VALID_RANGE = (2918, 3161)
TEST_RANGE = (3161, 3603)
EXPECTED_TEST_SHAPE = (442, 5282)
EXPECTED_TEST_EVAL_COUNT = 2_042_538
NEUTRAL_VALUE = np.float32(0.5)

assert TRAIN_RANGE == (TRAIN_START, VALID_START)
assert VALID_RANGE == (VALID_START, TEST_START)
assert TEST_RANGE == (TEST_START, T)
assert S == EXPECTED_TEST_SHAPE[1]

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP_ENABLED = DEVICE.type == "cuda"

ARTIFACTS = {
    "linear_model": OUTPUT_DIR / "linear_model_v2.npz",
    "valid_linear": OUTPUT_DIR / "valid_linear.npy",
    "test_linear": OUTPUT_DIR / "test_linear.npy",
    "tcn_model": OUTPUT_DIR / "tcn_model_v2.pt",
    "valid_tcn": OUTPUT_DIR / "valid_tcn.npy",
    "test_tcn": OUTPUT_DIR / "test_tcn.npy",
    "lgbm_model": OUTPUT_DIR / "lgbm_model_v2.txt",
    "valid_lgbm": OUTPUT_DIR / "valid_lgbm.npy",
    "test_lgbm": OUTPUT_DIR / "test_lgbm.npy",
    "weight_search": OUTPUT_DIR / "ensemble_weight_search.csv",
    "report": OUTPUT_DIR / "ensemble_report.md",
    "candidate": OUTPUT_DIR / "y1_ensemble.npy",
}

print(
    {
        "base_dir": str(BASE_DIR),
        "device": str(DEVICE),
        "force_retrain": FORCE_RETRAIN,
        "train": TRAIN_RANGE,
        "valid": VALID_RANGE,
        "test": TEST_RANGE,
    }
)


{'base_dir': 'D:\\google_dl\\book\\友安杯\\03_模型训练\\y1_pipeline_v2', 'device': 'cuda', 'force_retrain': False, 'train': (486, 2918), 'valid': (2918, 3161), 'test': (3161, 3603)}


D:\anaconda\anaconda_data\envs\jingge_ts\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. 数据与官方掩码

`CompetitionData` 通过 mmap 读取数据。这里复用 v2 已有的 `.cache_dp_v2/`，
避免重复解压约 9GB 的原始 pickle。


In [2]:
PIPELINE_CONFIG = PipelineConfig(
    data_path=str(ROOT_DIR / "data.z"),
    output_dir=str(OUTPUT_DIR),
    cache_dir=str(BASE_DIR / ".cache_dp_v2"),
    seed=SEED,
    final_train_stock_cap=1200,
    verbose=True,
)

data = CompetitionData(PIPELINE_CONFIG)

valid_mask = np.asarray(
    data.mask_x[VALID_START:TEST_START]
    & data.mask_y[VALID_START:TEST_START]
    & np.isfinite(data.y1[VALID_START:TEST_START]),
    dtype=bool,
)
test_mask = np.asarray(
    data.mask_x[TEST_START:T] & data.mask_y[TEST_START:T],
    dtype=bool,
)
valid_labels = np.asarray(data.y1[VALID_START:TEST_START], dtype=np.float32)

assert valid_mask.shape == (243, S)
assert test_mask.shape == EXPECTED_TEST_SHAPE
assert int(test_mask.sum()) == EXPECTED_TEST_EVAL_COUNT

print(
    {
        "valid_eval_count": int(valid_mask.sum()),
        "test_eval_count": int(test_mask.sum()),
        "data_cache": str(data.pickle_path),
    }
)


{'valid_eval_count': 982972, 'test_eval_count': 2042538, 'data_cache': 'D:\\google_dl\\book\\友安杯\\03_模型训练\\y1_pipeline_v2\\.cache_dp_v2\\payload.pkl'}


## 3. 通用验收、秩变换、RankIC 与权重搜索

所有模型的输出都先按官方掩码验收。截面秩协议固定为：

\[
r_i = \frac{\operatorname{rank}(x_i)-1}{n-1}
\]

并列值用平均秩；掩码外固定为 `0.5`。当某时点只有一个有效样本时，
该样本也填 `0.5`，避免除零。


In [3]:
def save_npy_atomic(path: Path, array: np.ndarray) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".partial.npy")
    np.save(temporary, array)
    temporary.replace(path)


def validate_prediction_array(
    predictions: np.ndarray,
    mask: np.ndarray,
    *,
    name: str,
    expected_eval_count: int | None = None,
) -> dict[str, object]:
    array = np.asarray(predictions)
    if array.shape != mask.shape:
        raise ValueError(f"{name}: shape {array.shape} != {mask.shape}")
    if array.dtype != np.float32:
        raise TypeError(f"{name}: dtype {array.dtype} != float32")
    if not np.all(np.isfinite(array)):
        raise ValueError(f"{name}: 存在 NaN/Inf")
    eval_count = int(mask.sum())
    if expected_eval_count is not None and eval_count != expected_eval_count:
        raise ValueError(
            f"{name}: eval_count {eval_count} != {expected_eval_count}"
        )
    if not np.all(array[~mask] == NEUTRAL_VALUE):
        bad = int(np.count_nonzero(array[~mask] != NEUTRAL_VALUE))
        raise ValueError(f"{name}: 掩码外有 {bad} 个值不等于 0.5")
    return {
        "name": name,
        "shape": tuple(array.shape),
        "dtype": str(array.dtype),
        "finite": True,
        "eval_count": eval_count,
        "neutral_count": int((~mask).sum()),
        "minimum_eval": float(array[mask].min()) if eval_count else math.nan,
        "maximum_eval": float(array[mask].max()) if eval_count else math.nan,
    }


def load_validated_prediction(
    path: Path,
    mask: np.ndarray,
    *,
    name: str,
    expected_eval_count: int | None = None,
) -> np.ndarray:
    array = np.load(path, allow_pickle=False)
    validate_prediction_array(
        array,
        mask,
        name=name,
        expected_eval_count=expected_eval_count,
    )
    return array


def prediction_artifact_is_valid(
    path: Path,
    mask: np.ndarray,
    *,
    name: str,
    expected_eval_count: int | None = None,
) -> bool:
    if not path.exists():
        return False
    try:
        load_validated_prediction(
            path,
            mask,
            name=name,
            expected_eval_count=expected_eval_count,
        )
        return True
    except Exception as error:
        print(f"{name} 验收失败，将重建：{error}")
        return False


def cross_sectional_percentile_rank(
    values: np.ndarray,
    mask: np.ndarray,
) -> np.ndarray:
    values = np.asarray(values)
    mask = np.asarray(mask, dtype=bool)
    if values.shape != mask.shape:
        raise ValueError("values 与 mask 形状不一致")
    output = np.full(values.shape, NEUTRAL_VALUE, dtype=np.float32)
    for time_idx in range(values.shape[0]):
        usable = mask[time_idx] & np.isfinite(values[time_idx])
        count = int(usable.sum())
        if count == 0:
            continue
        if count == 1:
            output[time_idx, usable] = NEUTRAL_VALUE
            continue
        ranks = rankdata(values[time_idx, usable], method="average")
        output[time_idx, usable] = (
            (ranks - 1.0) / (count - 1.0)
        ).astype(np.float32)
    return output


def rank_ic_by_time(
    predictions: np.ndarray,
    labels: np.ndarray,
    mask: np.ndarray,
) -> np.ndarray:
    predictions = np.asarray(predictions)
    labels = np.asarray(labels)
    mask = np.asarray(mask, dtype=bool)
    if predictions.shape != labels.shape or predictions.shape != mask.shape:
        raise ValueError("predictions、labels 与 mask 形状必须一致")
    scores = np.full(predictions.shape[0], np.nan, dtype=np.float64)
    for time_idx in range(predictions.shape[0]):
        usable = (
            mask[time_idx]
            & np.isfinite(predictions[time_idx])
            & np.isfinite(labels[time_idx])
        )
        if int(usable.sum()) < 2:
            continue
        pred_rank = rankdata(predictions[time_idx, usable], method="average")
        label_rank = rankdata(labels[time_idx, usable], method="average")
        if pred_rank.std() <= 0 or label_rank.std() <= 0:
            continue
        scores[time_idx] = np.corrcoef(pred_rank, label_rank)[0, 1]
    return scores


def mean_rank_ic(
    predictions: np.ndarray,
    labels: np.ndarray,
    mask: np.ndarray,
) -> float:
    scores = rank_ic_by_time(predictions, labels, mask)
    if not np.any(np.isfinite(scores)):
        return float("nan")
    return float(np.nanmean(scores))


def validate_weights(weights: np.ndarray) -> np.ndarray:
    weights = np.asarray(weights, dtype=np.float64)
    if weights.shape != (3,):
        raise ValueError("三模型权重必须形如 (3,)")
    if not np.all(np.isfinite(weights)):
        raise ValueError("权重必须为有限值")
    if np.any(weights < -1e-12):
        raise ValueError("权重必须非负")
    if not np.isclose(weights.sum(), 1.0, atol=1e-10):
        raise ValueError(f"权重和必须为 1，当前为 {weights.sum()}")
    return weights


def weight_grid(step: float = 0.05) -> np.ndarray:
    units = int(round(1.0 / step))
    if not np.isclose(units * step, 1.0):
        raise ValueError("step 必须整除 1")
    rows = []
    for linear_units in range(units + 1):
        for tcn_units in range(units - linear_units + 1):
            lgbm_units = units - linear_units - tcn_units
            rows.append(
                [
                    linear_units / units,
                    tcn_units / units,
                    lgbm_units / units,
                ]
            )
    return np.asarray(rows, dtype=np.float64)


def combine_ranked_predictions(
    ranked_predictions: list[np.ndarray],
    weights: np.ndarray,
    mask: np.ndarray,
) -> np.ndarray:
    weights = validate_weights(weights)
    if len(ranked_predictions) != 3:
        raise ValueError("必须提供三份秩预测")
    combined = np.full(mask.shape, NEUTRAL_VALUE, dtype=np.float32)
    weighted = np.zeros(mask.shape, dtype=np.float64)
    for weight, prediction in zip(weights, ranked_predictions):
        if prediction.shape != mask.shape:
            raise ValueError("秩预测形状不一致")
        weighted[mask] += weight * prediction[mask]
    combined[mask] = weighted[mask].astype(np.float32)
    return combined


def _weight_key(weights: np.ndarray) -> tuple[int, int, int]:
    return tuple(np.rint(np.asarray(weights) * 20).astype(int).tolist())


def search_ensemble_weights(
    ranked_valid: list[np.ndarray],
    labels: np.ndarray,
    mask: np.ndarray,
) -> tuple[pd.DataFrame, dict[str, float], pd.DataFrame]:
    if mask.shape[0] != 243:
        raise ValueError("官方 Valid 应为 243 个时点")
    model_names = ["linear", "tcn", "lgbm"]
    single_rows = []
    for name, prediction in zip(model_names, ranked_valid):
        scores = rank_ic_by_time(prediction, labels, mask)
        single_rows.append(
            {
                "model": name,
                "first_half_rank_ic": float(np.nanmean(scores[:122])),
                "second_half_rank_ic": float(np.nanmean(scores[122:])),
                "full_rank_ic": float(np.nanmean(scores)),
            }
        )
    single_table = pd.DataFrame(single_rows)
    best_single = {
        "first": float(single_table["first_half_rank_ic"].max()),
        "second": float(single_table["second_half_rank_ic"].max()),
        "full": float(single_table["full_rank_ic"].max()),
    }

    rows = []
    for weights in weight_grid(0.05):
        combined = combine_ranked_predictions(ranked_valid, weights, mask)
        scores = rank_ic_by_time(combined, labels, mask)
        rows.append(
            {
                "linear_weight": weights[0],
                "tcn_weight": weights[1],
                "lgbm_weight": weights[2],
                "first_half_rank_ic": float(np.nanmean(scores[:122])),
                "second_half_rank_ic": float(np.nanmean(scores[122:])),
                "full_rank_ic": float(np.nanmean(scores)),
            }
        )
    table = pd.DataFrame(rows)
    lookup = {
        _weight_key(
            row[["linear_weight", "tcn_weight", "lgbm_weight"]].to_numpy()
        ): float(row["full_rank_ic"])
        for _, row in table.iterrows()
    }

    plateau_deltas = []
    for _, row in table.iterrows():
        weights = row[
            ["linear_weight", "tcn_weight", "lgbm_weight"]
        ].to_numpy(dtype=np.float64)
        center = float(row["full_rank_ic"])
        neighbor_scores = []
        for donor in range(3):
            for receiver in range(3):
                if donor == receiver:
                    continue
                neighbor = weights.copy()
                neighbor[donor] -= 0.10
                neighbor[receiver] += 0.10
                if np.all(neighbor >= -1e-12) and np.all(neighbor <= 1 + 1e-12):
                    key = _weight_key(neighbor)
                    if key in lookup:
                        neighbor_scores.append(lookup[key])
        plateau_deltas.append(
            max(abs(score - center) for score in neighbor_scores)
            if neighbor_scores
            else math.inf
        )
    table["plateau_max_delta"] = plateau_deltas
    table["beats_first_half_single"] = (
        table["first_half_rank_ic"] > best_single["first"]
    )
    table["beats_second_half_single"] = (
        table["second_half_rank_ic"] > best_single["second"]
    )
    table["second_half_above_0_093940"] = (
        table["second_half_rank_ic"] > 0.093940
    )
    table["stable_plateau"] = table["plateau_max_delta"] < 0.001
    table["passes_all_gates"] = (
        table["beats_first_half_single"]
        & table["beats_second_half_single"]
        & table["second_half_above_0_093940"]
        & table["stable_plateau"]
    )
    table = table.sort_values(
        ["passes_all_gates", "full_rank_ic"],
        ascending=[False, False],
    ).reset_index(drop=True)
    return table, best_single, single_table


## 4. 短时合成自检

这部分不读取正式训练标签，不进入任何 epoch。它覆盖 tie、空/单点掩码、
权重和、RankIC、dtype 和掩码外中性值。


In [4]:
toy_values = np.array(
    [[3.0, 1.0, 1.0, 99.0], [5.0, 4.0, 3.0, 2.0]],
    dtype=np.float32,
)
toy_mask = np.array(
    [[True, True, True, False], [False, False, False, False]]
)
toy_rank = cross_sectional_percentile_rank(toy_values, toy_mask)
assert toy_rank.dtype == np.float32
assert np.allclose(toy_rank[0], [1.0, 0.25, 0.25, 0.5])
assert np.all(toy_rank[1] == 0.5)

one_mask = np.array([[False, True, False]])
one_rank = cross_sectional_percentile_rank(
    np.array([[9.0, 3.0, 1.0]], dtype=np.float32),
    one_mask,
)
assert np.all(one_rank == 0.5)

perfect_labels = np.array([[0.1, 0.2, 0.3, np.nan]], dtype=np.float32)
perfect_pred = np.array([[1.0, 2.0, 3.0, 0.5]], dtype=np.float32)
perfect_mask = np.array([[True, True, True, False]])
assert np.isclose(mean_rank_ic(perfect_pred, perfect_labels, perfect_mask), 1.0)

grid = weight_grid(0.05)
assert grid.shape == (231, 3)
assert np.all(grid >= 0)
assert np.allclose(grid.sum(axis=1), 1.0)
validate_weights(np.array([0.2, 0.3, 0.5]))

toy_combined = combine_ranked_predictions(
    [toy_rank, toy_rank, toy_rank],
    np.array([1 / 3, 1 / 3, 1 / 3]),
    toy_mask,
)
toy_report = validate_prediction_array(
    toy_combined,
    toy_mask,
    name="toy_combined",
)
assert toy_report["dtype"] == "float32"
print("通用合成自检通过", toy_report)


通用合成自检通过 {'name': 'toy_combined', 'shape': (2, 4), 'dtype': 'float32', 'finite': True, 'eval_count': 3, 'neutral_count': 5, 'minimum_eval': 0.25, 'maximum_eval': 1.0}


## 5. 线性模型：定义与微型损失测试

标准化参数只使用 Train `[486,2918)`。训练函数没有 Valid 标签参数。


In [5]:
LINEAR_CONFIG = {
    "epochs": 3,
    "time_chunk_size": 16,
    "batch_size": 16_384,
    "learning_rate": 0.02,
    "l2": 1e-5,
    "variance_floor": 1e-6,
    "seed": SEED,
}


def get_labeled_linear_chunk(
    source: CompetitionData,
    start: int,
    stop: int,
) -> tuple[np.ndarray, np.ndarray]:
    features = np.asarray(source.num_x[start:stop], dtype=np.float32)
    targets = np.asarray(source.y1[start:stop], dtype=np.float32)
    usable = (
        np.asarray(source.mask_x[start:stop])
        & np.asarray(source.mask_y[start:stop])
        & np.isfinite(targets)
        & np.all(np.isfinite(features), axis=2)
    )
    return features[usable], targets[usable]


def fit_linear_standardizer(
    source: CompetitionData,
    train_start: int,
    train_stop: int,
) -> tuple[np.ndarray, np.ndarray]:
    feature_sum = np.zeros(99, dtype=np.float64)
    squared_sum = np.zeros(99, dtype=np.float64)
    sample_count = 0
    for chunk_start in range(
        train_start, train_stop, LINEAR_CONFIG["time_chunk_size"]
    ):
        chunk_stop = min(
            chunk_start + LINEAR_CONFIG["time_chunk_size"], train_stop
        )
        features, _ = get_labeled_linear_chunk(
            source, chunk_start, chunk_stop
        )
        if not features.size:
            continue
        feature_sum += features.sum(axis=0, dtype=np.float64)
        squared_sum += np.square(features, dtype=np.float64).sum(axis=0)
        sample_count += features.shape[0]
    if sample_count == 0:
        raise RuntimeError("线性模型 Train 区间没有样本")
    mean = feature_sum / sample_count
    variance = squared_sum / sample_count - np.square(mean)
    std = np.sqrt(
        np.maximum(variance, LINEAR_CONFIG["variance_floor"])
    )
    return mean.astype(np.float32), std.astype(np.float32)


def update_linear_batch(
    features: np.ndarray,
    targets: np.ndarray,
    weights: np.ndarray,
    bias: np.float32,
    learning_rate: float,
) -> tuple[np.ndarray, np.float32, float, int]:
    predictions = features @ weights + bias
    error = predictions - targets
    count = features.shape[0]
    weight_gradient = (
        2.0 * features.T @ error / count
        + 2.0 * LINEAR_CONFIG["l2"] * weights
    )
    bias_gradient = 2.0 * error.mean()
    new_weights = (
        weights - learning_rate * weight_gradient
    ).astype(np.float32)
    new_bias = np.float32(bias - learning_rate * bias_gradient)
    return new_weights, new_bias, float(np.square(error).sum()), int(count)


def train_linear_once(
    source: CompetitionData,
) -> tuple[np.ndarray, np.float32, np.ndarray, np.ndarray, list[dict]]:
    feature_mean, feature_std = fit_linear_standardizer(
        source, TRAIN_START, VALID_START
    )
    rng = np.random.default_rng(LINEAR_CONFIG["seed"])
    weights = np.zeros(99, dtype=np.float32)
    bias = np.float32(0.5)
    history = []
    for epoch_idx in range(LINEAR_CONFIG["epochs"]):
        learning_rate = LINEAR_CONFIG["learning_rate"] / np.sqrt(epoch_idx + 1)
        chunk_starts = np.arange(
            TRAIN_START,
            VALID_START,
            LINEAR_CONFIG["time_chunk_size"],
        )
        rng.shuffle(chunk_starts)
        squared_error = 0.0
        sample_count = 0
        for chunk_start in chunk_starts:
            chunk_stop = min(
                int(chunk_start) + LINEAR_CONFIG["time_chunk_size"],
                VALID_START,
            )
            features, targets = get_labeled_linear_chunk(
                source, int(chunk_start), chunk_stop
            )
            if not features.size:
                continue
            features = (features - feature_mean) / feature_std
            order = rng.permutation(features.shape[0])
            for batch_start in range(
                0, features.shape[0], LINEAR_CONFIG["batch_size"]
            ):
                indices = order[
                    batch_start : batch_start + LINEAR_CONFIG["batch_size"]
                ]
                weights, bias, batch_error, batch_count = update_linear_batch(
                    features[indices],
                    targets[indices],
                    weights,
                    bias,
                    learning_rate,
                )
                squared_error += batch_error
                sample_count += batch_count
        if sample_count == 0:
            raise RuntimeError("线性模型 epoch 没有样本")
        row = {
            "epoch": epoch_idx + 1,
            "learning_rate": float(learning_rate),
            "train_mse": squared_error / sample_count,
            "train_samples": sample_count,
        }
        history.append(row)
        print("Linear", row)
    return weights, bias, feature_mean, feature_std, history


def predict_linear_range(
    source: CompetitionData,
    start: int,
    stop: int,
    mask: np.ndarray,
    weights: np.ndarray,
    bias: np.float32,
    feature_mean: np.ndarray,
    feature_std: np.ndarray,
) -> np.ndarray:
    output = np.full((stop - start, S), NEUTRAL_VALUE, dtype=np.float32)
    for local_idx, time_idx in enumerate(range(start, stop)):
        usable = mask[local_idx]
        if np.any(usable):
            features = np.asarray(
                source.num_x[time_idx, usable], dtype=np.float32
            )
            output[local_idx, usable] = (
                (features - feature_mean) / feature_std
            ) @ weights + bias
    return output


micro_features = np.zeros((4, 99), dtype=np.float32)
micro_targets = np.full(4, 0.5, dtype=np.float32)
micro_weights, micro_bias, micro_loss, micro_count = update_linear_batch(
    micro_features,
    micro_targets,
    np.zeros(99, dtype=np.float32),
    np.float32(0.5),
    0.02,
)
assert micro_count == 4 and micro_loss == 0.0
assert np.all(micro_weights == 0) and micro_bias == np.float32(0.5)
print("线性模型微型前向/损失测试通过")


线性模型微型前向/损失测试通过


## 6. 线性模型：训练/加载与 Valid/Test 推理

模型缺失或验收失败时训练一次；该模型同时预测 Valid 和 Test。


In [6]:
def load_linear_model(path: Path):
    artifact = np.load(path, allow_pickle=False)
    required = {"weights", "bias", "feature_mean", "feature_std"}
    if not required.issubset(artifact.files):
        raise ValueError(f"线性模型缺少键：{required - set(artifact.files)}")
    weights = artifact["weights"].astype(np.float32)
    bias = np.float32(artifact["bias"].item())
    mean = artifact["feature_mean"].astype(np.float32)
    std = artifact["feature_std"].astype(np.float32)
    if weights.shape != (99,) or mean.shape != (99,) or std.shape != (99,):
        raise ValueError("线性模型参数形状错误")
    if not (
        np.all(np.isfinite(weights))
        and np.isfinite(bias)
        and np.all(np.isfinite(mean))
        and np.all(np.isfinite(std))
        and np.all(std > 0)
    ):
        raise ValueError("线性模型参数非法")
    return weights, bias, mean, std


linear_model_valid = False
if ARTIFACTS["linear_model"].exists() and not FORCE_RETRAIN:
    try:
        linear_parameters = load_linear_model(ARTIFACTS["linear_model"])
        linear_model_valid = True
    except Exception as error:
        print("线性模型验收失败，将重训：", error)

linear_predictions_valid = (
    not FORCE_RETRAIN
    and prediction_artifact_is_valid(
        ARTIFACTS["valid_linear"], valid_mask, name="valid_linear"
    )
    and prediction_artifact_is_valid(
        ARTIFACTS["test_linear"],
        test_mask,
        name="test_linear",
        expected_eval_count=EXPECTED_TEST_EVAL_COUNT,
    )
)

linear_trained_now = False
if not linear_model_valid:
    (
        linear_weights,
        linear_bias,
        linear_mean,
        linear_std,
        linear_history,
    ) = train_linear_once(data)
    np.savez(
        ARTIFACTS["linear_model"],
        weights=linear_weights,
        bias=np.asarray(linear_bias, dtype=np.float32),
        feature_mean=linear_mean,
        feature_std=linear_std,
        train_start=np.int32(TRAIN_START),
        train_stop=np.int32(VALID_START),
        epochs=np.int32(LINEAR_CONFIG["epochs"]),
        seed=np.int32(SEED),
        training_history_json=np.asarray(json.dumps(linear_history)),
    )
    linear_parameters = (
        linear_weights,
        linear_bias,
        linear_mean,
        linear_std,
    )
    linear_trained_now = True

if linear_trained_now or not linear_predictions_valid:
    linear_weights, linear_bias, linear_mean, linear_std = linear_parameters
    valid_linear = predict_linear_range(
        data,
        VALID_START,
        TEST_START,
        valid_mask,
        linear_weights,
        linear_bias,
        linear_mean,
        linear_std,
    )
    test_linear = predict_linear_range(
        data,
        TEST_START,
        T,
        test_mask,
        linear_weights,
        linear_bias,
        linear_mean,
        linear_std,
    )
    validate_prediction_array(valid_linear, valid_mask, name="valid_linear")
    validate_prediction_array(
        test_linear,
        test_mask,
        name="test_linear",
        expected_eval_count=EXPECTED_TEST_EVAL_COUNT,
    )
    save_npy_atomic(ARTIFACTS["valid_linear"], valid_linear)
    save_npy_atomic(ARTIFACTS["test_linear"], test_linear)
else:
    valid_linear = load_validated_prediction(
        ARTIFACTS["valid_linear"], valid_mask, name="valid_linear"
    )
    test_linear = load_validated_prediction(
        ARTIFACTS["test_linear"],
        test_mask,
        name="test_linear",
        expected_eval_count=EXPECTED_TEST_EVAL_COUNT,
    )

print("线性模型阶段完成")


线性模型阶段完成


## 7. TCN：窗口、网络与微型前向/损失测试

TCN 使用 486 期因果窗口、99 个数值通道和 1 个有效性通道。
标准化统计只取 Train `[486,2918)`，固定训练 8 epochs；
官方 Valid 不用于早停、checkpoint 选择或任何训练决策。


In [7]:
@dataclass(frozen=True)
class TCNConfig:
    window_size: int = 486
    feature_count: int = 99
    input_channels: int = 100
    hidden_channels: int = 32
    dilations: tuple[int, ...] = (1, 2, 4, 8, 16, 32, 64, 128)
    dropout: float = 0.1
    train_time_bins: int = 64
    times_per_bin: int = 8
    stocks_per_time: int = 200
    batch_size: int = 128
    epochs: int = 8
    learning_rate: float = 1e-3
    weight_decay: float = 1e-4
    gradient_clip: float = 1.0
    statistics_time_chunk: int = 16
    variance_floor: float = 1e-6
    seed: int = SEED


TCN_CONFIG = TCNConfig()


def tcn_window_bounds(time_idx: int) -> tuple[int, int]:
    start = time_idx - TCN_CONFIG.window_size + 1
    stop = time_idx + 1
    if start < 0 or stop - start != TCN_CONFIG.window_size:
        raise ValueError(f"非法 TCN 时点：{time_idx}")
    return start, stop


def prepare_tcn_window_arrays(
    feature_window: np.ndarray,
    valid_window: np.ndarray,
    normalization_mean: np.ndarray,
    normalization_std: np.ndarray,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    feature_window = np.asarray(feature_window, dtype=np.float32)
    valid_window = np.asarray(valid_window, dtype=bool)
    valid_counts = valid_window.sum(axis=0, dtype=np.int32)
    nonempty = valid_counts > 0
    denominators = np.maximum(valid_counts, 1).astype(np.float32)[:, None]
    feature_sums = np.sum(
        feature_window,
        axis=0,
        where=valid_window[:, :, None],
        dtype=np.float64,
    ).astype(np.float32)
    window_means = feature_sums / denominators
    window_means[~nonempty] = normalization_mean
    filled = np.where(
        valid_window[:, :, None],
        feature_window,
        window_means[None, :, :],
    )
    standardized = (
        filled - normalization_mean[None, None, :]
    ) / normalization_std[None, None, :]
    mask_channel = valid_window[:, :, None].astype(np.float32)
    inputs = np.concatenate(
        [standardized, mask_channel], axis=2
    ).transpose(1, 2, 0)
    coverage = (
        valid_counts.astype(np.float32) / feature_window.shape[0]
    )[:, None]
    return (
        np.ascontiguousarray(inputs, dtype=np.float32),
        np.ascontiguousarray(coverage, dtype=np.float32),
        nonempty,
    )


class CausalResidualBlock(nn.Module):
    def __init__(self, channels: int, dilation: int, dropout: float):
        super().__init__()
        self.left_padding = nn.ConstantPad1d((2 * dilation, 0), 0.0)
        self.convolution = nn.Conv1d(
            channels, channels, kernel_size=3, dilation=dilation
        )
        self.normalization = nn.GroupNorm(4, channels)
        self.activation = nn.GELU()
        self.dropout = nn.Dropout(dropout)

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        hidden = self.left_padding(inputs)
        hidden = self.convolution(hidden)
        hidden = self.normalization(hidden)
        hidden = self.activation(hidden)
        hidden = self.dropout(hidden)
        return inputs + hidden


class WindowTCN(nn.Module):
    def __init__(
        self,
        input_channels: int,
        hidden_channels: int,
        dilations: tuple[int, ...],
        dropout: float,
    ):
        super().__init__()
        self.projection = nn.Conv1d(
            input_channels, hidden_channels, kernel_size=1
        )
        self.projection_activation = nn.GELU()
        self.blocks = nn.ModuleList(
            [
                CausalResidualBlock(hidden_channels, dilation, dropout)
                for dilation in dilations
            ]
        )
        self.head = nn.Sequential(
            nn.Linear(hidden_channels + 1, hidden_channels),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_channels, 1),
            nn.Sigmoid(),
        )

    def forward(
        self,
        sequence_inputs: torch.Tensor,
        coverage: torch.Tensor,
    ) -> torch.Tensor:
        hidden = self.projection_activation(self.projection(sequence_inputs))
        for block in self.blocks:
            hidden = block(hidden)
        final_hidden = hidden[:, :, -1]
        return self.head(torch.cat([final_hidden, coverage], dim=1))


def build_tcn_model() -> WindowTCN:
    return WindowTCN(
        TCN_CONFIG.input_channels,
        TCN_CONFIG.hidden_channels,
        TCN_CONFIG.dilations,
        TCN_CONFIG.dropout,
    )


tiny_tcn = WindowTCN(5, 8, (1, 2), 0.0)
tiny_sequence = torch.zeros(3, 5, 16)
tiny_coverage = torch.ones(3, 1)
tiny_target = torch.full((3, 1), 0.5)
tiny_output = tiny_tcn(tiny_sequence, tiny_coverage)
tiny_tcn_loss = nn.MSELoss()(tiny_output, tiny_target)
assert tiny_output.shape == (3, 1)
assert torch.isfinite(tiny_output).all() and torch.isfinite(tiny_tcn_loss)
del tiny_tcn, tiny_sequence, tiny_coverage, tiny_target, tiny_output
print("TCN 微型前向/损失测试通过")


TCN 微型前向/损失测试通过


## 8. TCN：训练/加载与 Valid/Test 推理

保存的是第 8 个 epoch 的最终模型，不基于 Valid 选择 checkpoint。


In [8]:
def fit_tcn_standardizer(
    source: CompetitionData,
) -> tuple[np.ndarray, np.ndarray]:
    feature_sum = np.zeros(TCN_CONFIG.feature_count, dtype=np.float64)
    squared_sum = np.zeros(TCN_CONFIG.feature_count, dtype=np.float64)
    sample_count = 0
    for chunk_start in range(
        TRAIN_START,
        VALID_START,
        TCN_CONFIG.statistics_time_chunk,
    ):
        chunk_stop = min(
            chunk_start + TCN_CONFIG.statistics_time_chunk,
            VALID_START,
        )
        feature_block = np.asarray(
            source.num_x[chunk_start:chunk_stop], dtype=np.float32
        )
        valid_block = np.asarray(
            source.mask_x[chunk_start:chunk_stop], dtype=bool
        )
        valid_features = feature_block[valid_block]
        feature_sum += valid_features.sum(axis=0, dtype=np.float64)
        squared_sum += np.square(
            valid_features, dtype=np.float64
        ).sum(axis=0)
        sample_count += valid_features.shape[0]
    if sample_count == 0:
        raise RuntimeError("TCN Train 区间没有标准化样本")
    mean = feature_sum / sample_count
    variance = squared_sum / sample_count - np.square(mean)
    std = np.sqrt(np.maximum(variance, TCN_CONFIG.variance_floor))
    return mean.astype(np.float32), std.astype(np.float32)


def build_tcn_batch(
    source: CompetitionData,
    time_idx: int,
    stocks: np.ndarray,
    feature_mean: np.ndarray,
    feature_std: np.ndarray,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    start, stop = tcn_window_bounds(time_idx)
    return prepare_tcn_window_arrays(
        source.num_x[start:stop, stocks, :],
        source.mask_x[start:stop, stocks],
        feature_mean,
        feature_std,
    )


def sample_tcn_training_times(rng: np.random.Generator) -> np.ndarray:
    bins = np.array_split(
        np.arange(TRAIN_START, VALID_START, dtype=np.int64),
        TCN_CONFIG.train_time_bins,
    )
    selected = []
    for time_bin in bins:
        count = min(TCN_CONFIG.times_per_bin, time_bin.size)
        selected.extend(
            rng.choice(time_bin, size=count, replace=False).tolist()
        )
    selected = np.asarray(selected, dtype=np.int64)
    rng.shuffle(selected)
    return selected


def train_tcn_once(
    source: CompetitionData,
) -> tuple[WindowTCN, np.ndarray, np.ndarray, list[dict]]:
    feature_mean, feature_std = fit_tcn_standardizer(source)
    model = build_tcn_model().to(DEVICE)
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=TCN_CONFIG.learning_rate,
        weight_decay=TCN_CONFIG.weight_decay,
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=TCN_CONFIG.epochs
    )
    loss_function = nn.MSELoss()
    gradient_scaler = torch.amp.GradScaler(
        "cuda", enabled=AMP_ENABLED
    )
    rng = np.random.default_rng(TCN_CONFIG.seed)
    history = []

    for epoch_idx in range(TCN_CONFIG.epochs):
        model.train()
        squared_error = 0.0
        sample_count = 0
        for time_idx in sample_tcn_training_times(rng):
            time_idx = int(time_idx)
            stocks = source.eligible_stocks(time_idx, require_label=True)
            if stocks.size > TCN_CONFIG.stocks_per_time:
                stocks = rng.choice(
                    stocks,
                    size=TCN_CONFIG.stocks_per_time,
                    replace=False,
                )
            stocks = np.asarray(stocks, dtype=np.int64)
            rng.shuffle(stocks)
            for batch_start in range(
                0, stocks.size, TCN_CONFIG.batch_size
            ):
                batch_stocks = stocks[
                    batch_start : batch_start + TCN_CONFIG.batch_size
                ]
                inputs, coverage, nonempty = build_tcn_batch(
                    source,
                    time_idx,
                    batch_stocks,
                    feature_mean,
                    feature_std,
                )
                if not np.all(nonempty):
                    positions = np.flatnonzero(nonempty)
                    batch_stocks = batch_stocks[positions]
                    inputs = inputs[positions]
                    coverage = coverage[positions]
                if batch_stocks.size == 0:
                    continue
                targets = np.asarray(
                    source.y1[time_idx, batch_stocks],
                    dtype=np.float32,
                )[:, None]
                input_tensor = torch.from_numpy(inputs).to(
                    DEVICE, non_blocking=True
                )
                coverage_tensor = torch.from_numpy(coverage).to(
                    DEVICE, non_blocking=True
                )
                target_tensor = torch.from_numpy(targets).to(
                    DEVICE, non_blocking=True
                )
                optimizer.zero_grad(set_to_none=True)
                with torch.autocast(
                    device_type=DEVICE.type,
                    dtype=torch.float16,
                    enabled=AMP_ENABLED,
                ):
                    predictions = model(input_tensor, coverage_tensor)
                    loss = loss_function(predictions, target_tensor)
                gradient_scaler.scale(loss).backward()
                gradient_scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(
                    model.parameters(), TCN_CONFIG.gradient_clip
                )
                gradient_scaler.step(optimizer)
                gradient_scaler.update()
                count = batch_stocks.size
                squared_error += float(loss.item()) * count
                sample_count += count
        if sample_count == 0:
            raise RuntimeError("TCN epoch 没有训练样本")
        current_lr = optimizer.param_groups[0]["lr"]
        scheduler.step()
        row = {
            "epoch": epoch_idx + 1,
            "learning_rate": float(current_lr),
            "train_samples": int(sample_count),
            "train_mse": squared_error / sample_count,
        }
        history.append(row)
        print("TCN", row)
    return model, feature_mean, feature_std, history


def save_tcn_checkpoint(
    path: Path,
    model: WindowTCN,
    feature_mean: np.ndarray,
    feature_std: np.ndarray,
    history: list[dict],
) -> None:
    temporary = path.with_name(path.name + ".partial")
    torch.save(
        {
            "model_state_dict": {
                name: value.detach().cpu()
                for name, value in model.state_dict().items()
            },
            "feature_mean": torch.from_numpy(feature_mean.copy()),
            "feature_std": torch.from_numpy(feature_std.copy()),
            "config": asdict(TCN_CONFIG),
            "train_start": TRAIN_START,
            "train_stop": VALID_START,
            "completed_epochs": TCN_CONFIG.epochs,
            "training_history": history,
        },
        temporary,
    )
    temporary.replace(path)


def load_tcn_checkpoint(
    path: Path,
) -> tuple[WindowTCN, np.ndarray, np.ndarray, list[dict]]:
    checkpoint = torch.load(path, map_location="cpu", weights_only=False)
    if checkpoint.get("completed_epochs") != TCN_CONFIG.epochs:
        raise ValueError("TCN checkpoint 不是完整 8 epochs")
    if (
        checkpoint.get("train_start") != TRAIN_START
        or checkpoint.get("train_stop") != VALID_START
    ):
        raise ValueError("TCN checkpoint 训练边界不一致")
    expected_config = asdict(TCN_CONFIG)
    stored_config = checkpoint.get("config", {})
    if stored_config != expected_config:
        raise ValueError("TCN checkpoint 配置不一致")
    model = build_tcn_model()
    model.load_state_dict(checkpoint["model_state_dict"], strict=True)
    model = model.to(DEVICE)
    feature_mean = np.asarray(
        checkpoint["feature_mean"], dtype=np.float32
    )
    feature_std = np.asarray(
        checkpoint["feature_std"], dtype=np.float32
    )
    if feature_mean.shape != (99,) or feature_std.shape != (99,):
        raise ValueError("TCN 标准化参数形状错误")
    if not np.all(np.isfinite(feature_mean)) or not np.all(feature_std > 0):
        raise ValueError("TCN 标准化参数非法")
    return (
        model,
        feature_mean,
        feature_std,
        checkpoint.get("training_history", []),
    )


def predict_tcn_range(
    source: CompetitionData,
    model: WindowTCN,
    start: int,
    stop: int,
    mask: np.ndarray,
    feature_mean: np.ndarray,
    feature_std: np.ndarray,
) -> np.ndarray:
    output = np.full((stop - start, S), NEUTRAL_VALUE, dtype=np.float32)
    model.eval()
    for local_idx, time_idx in enumerate(range(start, stop)):
        stocks = np.flatnonzero(mask[local_idx])
        for batch_start in range(
            0, stocks.size, TCN_CONFIG.batch_size
        ):
            batch_stocks = stocks[
                batch_start : batch_start + TCN_CONFIG.batch_size
            ]
            inputs, coverage, nonempty = build_tcn_batch(
                source,
                time_idx,
                batch_stocks,
                feature_mean,
                feature_std,
            )
            if not np.any(nonempty):
                continue
            positions = np.flatnonzero(nonempty)
            input_tensor = torch.from_numpy(inputs[positions]).to(
                DEVICE, non_blocking=True
            )
            coverage_tensor = torch.from_numpy(coverage[positions]).to(
                DEVICE, non_blocking=True
            )
            with torch.inference_mode():
                with torch.autocast(
                    device_type=DEVICE.type,
                    dtype=torch.float16,
                    enabled=AMP_ENABLED,
                ):
                    predictions = model(input_tensor, coverage_tensor)
            output[
                local_idx, batch_stocks[positions]
            ] = predictions.squeeze(1).float().cpu().numpy()
        if (local_idx + 1) % 25 == 0 or time_idx == stop - 1:
            print(f"TCN inference {local_idx + 1}/{stop - start}")
    return output


tcn_model_valid = False
if ARTIFACTS["tcn_model"].exists() and not FORCE_RETRAIN:
    try:
        tcn_parameters = load_tcn_checkpoint(ARTIFACTS["tcn_model"])
        tcn_model_valid = True
    except Exception as error:
        print("TCN 模型验收失败，将重训：", error)

tcn_predictions_valid = (
    not FORCE_RETRAIN
    and prediction_artifact_is_valid(
        ARTIFACTS["valid_tcn"], valid_mask, name="valid_tcn"
    )
    and prediction_artifact_is_valid(
        ARTIFACTS["test_tcn"],
        test_mask,
        name="test_tcn",
        expected_eval_count=EXPECTED_TEST_EVAL_COUNT,
    )
)

tcn_trained_now = False
if not tcn_model_valid:
    tcn_model, tcn_mean, tcn_std, tcn_history = train_tcn_once(data)
    save_tcn_checkpoint(
        ARTIFACTS["tcn_model"],
        tcn_model,
        tcn_mean,
        tcn_std,
        tcn_history,
    )
    tcn_parameters = (tcn_model, tcn_mean, tcn_std, tcn_history)
    tcn_trained_now = True

if tcn_trained_now or not tcn_predictions_valid:
    tcn_model, tcn_mean, tcn_std, _ = tcn_parameters
    valid_tcn = predict_tcn_range(
        data,
        tcn_model,
        VALID_START,
        TEST_START,
        valid_mask,
        tcn_mean,
        tcn_std,
    )
    test_tcn = predict_tcn_range(
        data,
        tcn_model,
        TEST_START,
        T,
        test_mask,
        tcn_mean,
        tcn_std,
    )
    validate_prediction_array(valid_tcn, valid_mask, name="valid_tcn")
    validate_prediction_array(
        test_tcn,
        test_mask,
        name="test_tcn",
        expected_eval_count=EXPECTED_TEST_EVAL_COUNT,
    )
    save_npy_atomic(ARTIFACTS["valid_tcn"], valid_tcn)
    save_npy_atomic(ARTIFACTS["test_tcn"], test_tcn)
else:
    valid_tcn = load_validated_prediction(
        ARTIFACTS["valid_tcn"], valid_mask, name="valid_tcn"
    )
    test_tcn = load_validated_prediction(
        ARTIFACTS["test_tcn"],
        test_mask,
        name="test_tcn",
        expected_eval_count=EXPECTED_TEST_EVAL_COUNT,
    )

print("TCN 阶段完成")


TCN 阶段完成


## 9. LightGBM：固定配置与微型功能测试

使用既有 `parameter_verification.csv` 中 `parameter_set=0, fold_3`
的 `best_iteration=1558`，不重新早停。正式模型只在完整 Train 上训练一次。


In [9]:
MANIFEST_PATH = RANK_OUTPUT_DIR / "feature_manifest.json"
PARAMETER_VERIFICATION_PATH = (
    RANK_OUTPUT_DIR / "parameter_verification.csv"
)

manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
selected_numeric_features = np.asarray(
    manifest["selected_numeric_features"], dtype=np.int32
)
selected_history_features = np.asarray(
    manifest["selected_history_features"], dtype=np.int32
)
lgbm_params = dict(manifest["winning_lightgbm_params"])
lgbm_params.update(
    {
        "seed": SEED,
        "feature_fraction_seed": SEED,
        "bagging_seed": SEED,
    }
)
LGBM_FEATURE_PREFIX = 328
LGBM_BOOST_ROUNDS = 1558

verification = pd.read_csv(PARAMETER_VERIFICATION_PATH)
fold_3_record = verification[
    (verification["parameter_set"] == 0)
    & (verification["fold"] == "fold_3")
]
assert len(fold_3_record) == 1
assert int(fold_3_record.iloc[0]["best_iteration"]) == LGBM_BOOST_ROUNDS
assert selected_numeric_features.shape == (40,)
assert selected_history_features.shape == (20,)
assert int(manifest["feature_prefix"]) == LGBM_FEATURE_PREFIX

# 微型功能测试：两个 3 行 query、1 轮，仅验证 Dataset/排序目标/预测链路。
tiny_lgbm_x = np.array(
    [[0.0], [1.0], [2.0], [2.0], [1.0], [0.0]],
    dtype=np.float32,
)
tiny_lgbm_y = np.array([0, 1, 2, 2, 1, 0], dtype=np.int32)
tiny_lgbm_dataset = lgb.Dataset(
    tiny_lgbm_x,
    label=tiny_lgbm_y,
    group=[3, 3],
    free_raw_data=False,
)
tiny_lgbm_model = lgb.train(
    {
        "objective": "lambdarank",
        "metric": "None",
        "verbosity": -1,
        "num_threads": 1,
        "seed": SEED,
        "min_data_in_leaf": 1,
    },
    tiny_lgbm_dataset,
    num_boost_round=1,
    callbacks=[lgb.log_evaluation(0)],
)
tiny_lgbm_prediction = tiny_lgbm_model.predict(tiny_lgbm_x)
assert tiny_lgbm_prediction.shape == (6,)
assert np.all(np.isfinite(tiny_lgbm_prediction))
del tiny_lgbm_model, tiny_lgbm_dataset
print("LightGBM 微型 1 轮功能测试通过")


LightGBM 微型 1 轮功能测试通过

## 10. LightGBM：训练/加载与 Valid/Test 推理

40 个数值特征、20 个历史特征、趋势填充与 328 维 numeric 前缀
由现有特征库构造。训练矩阵严格为 `[486,2918)`；Valid 只用于训练完成后的评估。


In [10]:
def lgbm_model_is_valid(path: Path) -> tuple[bool, lgb.Booster | None]:
    if not path.exists():
        return False, None
    try:
        booster = load_booster(path)
        if booster.num_trees() != LGBM_BOOST_ROUNDS:
            raise ValueError(
                f"树数量 {booster.num_trees()} != {LGBM_BOOST_ROUNDS}"
            )
        return True, booster
    except Exception as error:
        print("LightGBM 模型验收失败，将重训：", error)
        return False, None


def predict_lgbm_matrix(
    source: CompetitionData,
    booster: lgb.Booster,
    matrix,
    start: int,
    stop: int,
    role: str,
) -> np.ndarray:
    values = matrix.open()
    raw = booster.predict(
        values[:, :LGBM_FEATURE_PREFIX],
        num_iteration=LGBM_BOOST_ROUNDS,
    )
    output = np.full((stop - start, S), NEUTRAL_VALUE, dtype=np.float32)
    offset = 0
    for local_idx, (time_idx, group_size) in enumerate(
        zip(range(start, stop), matrix.groups)
    ):
        stocks = deterministic_stock_sample(
            source.eligible_stocks(
                time_idx, require_label=(role != "test")
            ),
            None,
        )
        group_size = int(group_size)
        if stocks.size != group_size:
            raise AssertionError(
                f"{role} t={time_idx}: stocks {stocks.size} != group {group_size}"
            )
        output[local_idx, stocks] = raw[
            offset : offset + group_size
        ].astype(np.float32)
        offset += group_size
    if offset != raw.size:
        raise AssertionError(f"{role}: row mapping {offset} != {raw.size}")
    del values, raw
    return output


lgbm_model_valid, lgbm_model = (
    (False, None)
    if FORCE_RETRAIN
    else lgbm_model_is_valid(ARTIFACTS["lgbm_model"])
)
lgbm_predictions_valid = (
    not FORCE_RETRAIN
    and prediction_artifact_is_valid(
        ARTIFACTS["valid_lgbm"], valid_mask, name="valid_lgbm"
    )
    and prediction_artifact_is_valid(
        ARTIFACTS["test_lgbm"],
        test_mask,
        name="test_lgbm",
        expected_eval_count=EXPECTED_TEST_EVAL_COUNT,
    )
)

lgbm_trained_now = False
if not (lgbm_model_valid and lgbm_predictions_valid):
    history_paths = build_history_cache(
        data, selected_history_features, PIPELINE_CONFIG
    )
    history_cache = open_history_cache(
        history_paths, selected_history_features.size
    )
    category_state = fit_category_state(
        data,
        TRAIN_START,
        VALID_START,
        selected_history_features,
        PIPELINE_CONFIG,
    )

    train_matrix = None
    valid_matrix = None
    test_matrix = None
    try:
        if not lgbm_model_valid:
            train_matrix = build_feature_matrix(
                data=data,
                history_cache=history_cache,
                selected_features=selected_numeric_features,
                history_features=selected_history_features,
                interaction_specs=[],
                category_state=category_state,
                start=TRAIN_START,
                stop=VALID_START,
                stock_cap=PIPELINE_CONFIG.final_train_stock_cap,
                role="train",
                imputation_mode="trend",
                matrix_name="ensemble_lgbm_train",
                config=PIPELINE_CONFIG,
            )
            if train_matrix.block_ends["numeric"] != LGBM_FEATURE_PREFIX:
                raise AssertionError("LightGBM numeric 前缀不是 328")
            lgbm_model = train_ranker_no_validation(
                train_matrix,
                LGBM_FEATURE_PREFIX,
                lgbm_params,
                LGBM_BOOST_ROUNDS,
            )
            save_booster(lgbm_model, ARTIFACTS["lgbm_model"])
            lgbm_trained_now = True

        valid_matrix = build_feature_matrix(
            data=data,
            history_cache=history_cache,
            selected_features=selected_numeric_features,
            history_features=selected_history_features,
            interaction_specs=[],
            category_state=category_state,
            start=VALID_START,
            stop=TEST_START,
            stock_cap=None,
            role="valid",
            imputation_mode="trend",
            matrix_name="ensemble_lgbm_valid",
            config=PIPELINE_CONFIG,
        )
        test_matrix = build_feature_matrix(
            data=data,
            history_cache=history_cache,
            selected_features=selected_numeric_features,
            history_features=selected_history_features,
            interaction_specs=[],
            category_state=category_state,
            start=TEST_START,
            stop=T,
            stock_cap=None,
            role="test",
            imputation_mode="trend",
            matrix_name="ensemble_lgbm_test",
            config=PIPELINE_CONFIG,
        )
        if (
            valid_matrix.block_ends["numeric"] != LGBM_FEATURE_PREFIX
            or test_matrix.block_ends["numeric"] != LGBM_FEATURE_PREFIX
        ):
            raise AssertionError("Valid/Test numeric 前缀不是 328")

        valid_lgbm = predict_lgbm_matrix(
            data,
            lgbm_model,
            valid_matrix,
            VALID_START,
            TEST_START,
            "valid",
        )
        test_lgbm = predict_lgbm_matrix(
            data,
            lgbm_model,
            test_matrix,
            TEST_START,
            T,
            "test",
        )
        validate_prediction_array(valid_lgbm, valid_mask, name="valid_lgbm")
        validate_prediction_array(
            test_lgbm,
            test_mask,
            name="test_lgbm",
            expected_eval_count=EXPECTED_TEST_EVAL_COUNT,
        )
        save_npy_atomic(ARTIFACTS["valid_lgbm"], valid_lgbm)
        save_npy_atomic(ARTIFACTS["test_lgbm"], test_lgbm)
    finally:
        for matrix in (train_matrix, valid_matrix, test_matrix):
            if matrix is not None:
                remove_feature_matrix(matrix)
        del history_cache
        gc.collect()
else:
    valid_lgbm = load_validated_prediction(
        ARTIFACTS["valid_lgbm"], valid_mask, name="valid_lgbm"
    )
    test_lgbm = load_validated_prediction(
        ARTIFACTS["test_lgbm"],
        test_mask,
        name="test_lgbm",
        expected_eval_count=EXPECTED_TEST_EVAL_COUNT,
    )

print(
    "LightGBM 阶段完成",
    {"trained_now": lgbm_trained_now, "boost_rounds": LGBM_BOOST_ROUNDS},
)


LightGBM 阶段完成

 {'trained_now': False, 'boost_rounds': 1558}


## 11. Valid 评估与三模型权重搜索

先把三份 Valid 预测分别转换为逐时点截面秩，再搜索步长 0.05 的
231 组非负权重。前 122 期用于选择，后 121 期用于确认。


In [11]:
model_valid_predictions = [valid_linear, valid_tcn, valid_lgbm]
model_test_predictions = [test_linear, test_tcn, test_lgbm]
model_names = ["linear", "tcn", "lgbm"]

ranked_valid = [
    cross_sectional_percentile_rank(prediction, valid_mask)
    for prediction in model_valid_predictions
]
ranked_test = [
    cross_sectional_percentile_rank(prediction, test_mask)
    for prediction in model_test_predictions
]

weight_search_table, best_single, single_model_table = (
    search_ensemble_weights(ranked_valid, valid_labels, valid_mask)
)
weight_search_table.to_csv(ARTIFACTS["weight_search"], index=False)

display(single_model_table)
display(weight_search_table.head(20))
print("最佳单模型半窗/全窗：", best_single)


,model,first_half_rank_ic,second_half_rank_ic,full_rank_ic
0,linear,0.094542,0.084773,0.089678
1,tcn,0.105142,0.083234,0.094233
2,lgbm,0.071455,0.064819,0.068151


,linear_weight,tcn_weight,lgbm_weight,first_half_rank_ic,second_half_rank_ic,full_rank_ic,plateau_max_delta,beats_first_half_single,beats_second_half_single,second_half_above_0_093940,stable_plateau,passes_all_gates
0,0.40,0.40,0.20,0.109618,0.092883,0.101285,0.001277,True,True,False,False,False
1,0.45,0.40,0.15,0.109609,0.092866,0.101272,0.001164,True,True,False,False,False
2,0.40,0.45,0.15,0.109972,0.092495,0.101269,0.000839,True,True,False,True,False
3,0.45,0.35,0.20,0.109198,0.093224,0.101244,0.000987,True,True,False,True,False
4,0.35,0.45,0.20,0.109775,0.092326,0.101087,0.001638,True,True,False,False,False
5,0.50,0.35,0.15,0.108994,0.093053,0.101056,0.001464,True,True,False,False,False
6,0.35,0.50,0.15,0.110068,0.091935,0.101039,0.001136,True,True,False,False,False
7,0.45,0.45,0.10,0.109694,0.092299,0.101032,0.001542,True,True,False,False,False
8,0.40,0.35,0.25,0.108933,0.093026,0.101012,0.001853,True,True,False,False,False
9,0.40,0.50,0.10,0.110000,0.091900,0.100987,0.001218,True,True,False,False,False


最佳单模型半窗/全窗： {'first': 0.1051424106537783, 'second': 0.08477339784170095, 'full': 0.09423321261620927}


## 12. 晋级门槛、候选文件与报告

候选必须同时满足：

- 前半窗超过对应最佳单模型；
- 后半窗超过对应最佳单模型；
- 后半窗 RankIC 高于 `0.093940`；
- 可行的 `±0.1` 邻近权重，其完整 Valid RankIC 最大波动小于 `0.001`。

通过才写入 `outputs/y1_ensemble.npy`；否则只写实验报告。


In [12]:
passing = weight_search_table[weight_search_table["passes_all_gates"]]
promoted = not passing.empty
selected_row = passing.iloc[0] if promoted else weight_search_table.iloc[0]
selected_weights = selected_row[
    ["linear_weight", "tcn_weight", "lgbm_weight"]
].to_numpy(dtype=np.float64)

candidate_validation = None
if promoted:
    candidate = combine_ranked_predictions(
        ranked_test, selected_weights, test_mask
    )
    candidate_validation = validate_prediction_array(
        candidate,
        test_mask,
        name="y1_ensemble",
        expected_eval_count=EXPECTED_TEST_EVAL_COUNT,
    )
    if candidate.shape != EXPECTED_TEST_SHAPE:
        raise AssertionError(candidate.shape)
    save_npy_atomic(ARTIFACTS["candidate"], candidate)
    print("全部门槛通过，已生成：", ARTIFACTS["candidate"])
else:
    print(
        "没有权重组合通过全部门槛；本次不会创建或覆盖 y1_ensemble.npy。"
    )
    if ARTIFACTS["candidate"].exists():
        print(
            "注意：目录中已有历史 y1_ensemble.npy，本次未修改；"
            "请以本报告的 promoted=False 为准。"
        )

report_lines = [
    "# 三模型重训与秩集成实验报告",
    "",
    f"- 生成时间：{time.strftime('%Y-%m-%d %H:%M:%S')}",
    f"- Train：`[{TRAIN_START}, {VALID_START})`",
    f"- Valid：`[{VALID_START}, {TEST_START})`",
    f"- Test：`[{TEST_START}, {T})`",
    "- 线性模型：3 epochs，Train-only 标准化",
    "- TCN：486 期窗口，固定 8 epochs，无 Valid 早停",
    f"- LightGBM：固定 {LGBM_BOOST_ROUNDS} 轮，无 Valid 早停",
    f"- 权重网格：{len(weight_search_table)} 组，步长 0.05",
    "",
    "## 单模型 Valid RankIC",
    "",
    "```csv\n" + single_model_table.to_csv(index=False, float_format="%.6f").strip() + "\n```",
    "",
    "## 选择结果",
    "",
    f"- promoted：`{promoted}`",
    (
        "- 选定权重（linear / tcn / lgbm）："
        f"`{selected_weights[0]:.2f} / {selected_weights[1]:.2f} / "
        f"{selected_weights[2]:.2f}`"
    ),
    f"- 前半窗 RankIC：`{selected_row['first_half_rank_ic']:.6f}`",
    f"- 后半窗 RankIC：`{selected_row['second_half_rank_ic']:.6f}`",
    f"- 完整 Valid RankIC：`{selected_row['full_rank_ic']:.6f}`",
    f"- 邻近权重最大波动：`{selected_row['plateau_max_delta']:.6f}`",
    f"- 前半窗超过最佳单模型：`{bool(selected_row['beats_first_half_single'])}`",
    f"- 后半窗超过最佳单模型：`{bool(selected_row['beats_second_half_single'])}`",
    f"- 后半窗高于 0.093940：`{bool(selected_row['second_half_above_0_093940'])}`",
    f"- 权重平台稳定：`{bool(selected_row['stable_plateau'])}`",
    "",
    "## 输出",
    "",
    (
        f"- 候选文件：`{ARTIFACTS['candidate'].name}`"
        if promoted
        else "- 未通过门槛，因此本次未生成候选提交文件。"
    ),
    f"- 权重明细：`{ARTIFACTS['weight_search'].name}`",
]
if candidate_validation is not None:
    report_lines.extend(
        [
            f"- 候选形状：`{candidate_validation['shape']}`",
            f"- 候选 dtype：`{candidate_validation['dtype']}`",
            f"- 评估位数量：`{candidate_validation['eval_count']}`",
            "- 掩码外：全部严格等于 `0.5`",
        ]
    )

ARTIFACTS["report"].write_text(
    "\n".join(report_lines) + "\n", encoding="utf-8"
)
print("实验报告：", ARTIFACTS["report"])
print("当前已知最佳提交始终保留：", RANK_OUTPUT_DIR / "y1_best.npy")


没有权重组合通过全部门槛；本次不会创建或覆盖 y1_ensemble.npy。
实验报告： D:\google_dl\book\友安杯\03_模型训练\y1_pipeline_v2\outputs\ensemble_report.md
当前已知最佳提交始终保留： D:\google_dl\book\友安杯\03_模型训练\y1_rank_pipeline\y1_rank_outputs\y1_best.npy


## 13. 执行后检查

若 `promoted=True`，请最终确认 `outputs/y1_ensemble.npy` 的验收结果；
若 `promoted=False`，继续使用当前已知最佳
`../y1_rank_pipeline/y1_rank_outputs/y1_best.npy`。

本 Notebook 不会修改或覆盖该现有最佳文件。
